# Requirements

This notebook assumes the Oracle database already contains the ingested `research_papers` table, embeddings, and related search indexes.

It includes the setup needed to run the `Build an Agent with Multiple Tool Access` example.

In [2]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    user = "system"
    password = "OraclePwd_2025"  # must match ORACLE_PWD from docker run
    dsn = "localhost:1521/FREEPDB1"
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener is not bound to 0.0.0.0 (run fix_oracle_listener())")
                print("    3. Container is not running (check with check_docker_container())")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running:")
                    print("     1. check_docker_container() - verify container is running")
                    print("     2. fix_oracle_listener() - fix listener binding")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

# Connect to Oracle
conn = connect_to_oracle()

Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free


In [3]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


In [4]:
import numpy as np
import array

In [5]:
def keyword_search_research_papers(conn, keyword: str):
    """
    Perform a full-text keyword search on the 'text' column 
    using the Oracle Text index (rp_text_idx).

    Args:
        conn: Oracle database connection object.
        keyword (str): Keyword or phrase to search for.

    Returns:
        tuple: (rows, columns)
    """
    query = """
        SELECT 
            arxiv_id, 
            title, 
            SUBSTR(text, 1, 200) AS text_snippet,
            SCORE(1) AS relevance_score
        FROM research_papers
        WHERE CONTAINS(text, :keyword, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST 10 ROWS ONLY
    """

    with conn.cursor() as cur:
        cur.execute(query, keyword=keyword)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [6]:
SEARCH_QUERY = "Get me papers related to planetary exploration"

In [7]:
def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):
    """
    Perform a vector similarity search on the research_papers table using a query embedding.
    Returns cosine similarity scores (higher = more similar).
    """

    # 1️⃣ Encode the query into a vector
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()

    # 2️⃣ Prepare the vector for Oracle binding
    query_embedding_array = array.array('f', query_embedding)

    # 3️⃣ Run a vector similarity search using cosine similarity
    query = f"""
        SELECT 
            arxiv_id, 
            title, 
            abstract, 
            SUBSTR(text, 1, 200) AS text_snippet,
            ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
        FROM research_papers
        ORDER BY similarity_score DESC
        FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
    """

    # 4️⃣ Execute and return results
    with conn.cursor() as cur:
        cur.execute(query, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [8]:
import array
import numpy as np

def hybrid_search_research_papers_pre_filter(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    show_explain: bool = False
):
    """
    Perform a hybrid search using Oracle Text + Vector Search.
    Combines lexical filtering (CONTAINS) with semantic re-ranking via cosine similarity.

    Args:
        conn: Oracle database connection object.
        embedding_model: Model with `.encode()` method (e.g., SentenceTransformer).
        search_phrase (str): User search phrase used for both text filtering and embedding.
        top_k (int): Number of results to return (default = 10).
        show_explain (bool): If True, prints the execution plan.

    Returns:
        tuple: (rows, columns, exec_plan_text or None)
    """

    # --- Step 1: Encode search phrase into normalized vector ---
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        # Enable runtime stats if needed
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        # --- Step 2: Hybrid query (Oracle Text + Vector) ---
        sql = f"""
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
                arxiv_id,
                title,
                abstract,
                SUBSTR(text, 1, 200) AS text_snippet,
                ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
            FROM research_papers
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
        """

        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    # --- Step 3: Execution plan (optional) ---
    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())

        print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
        print(exec_plan_text)
        print("========================================================\n")

    return rows, columns, exec_plan_text


In [9]:
# Azure OpenAI environment setup helpers
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely_azure(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [10]:
# https://azure-agent-ai-foundry-resource.openai.azure.com/
# gpt-4o
# gpt-4.1
# gpt-5

set_env_securely_azure("AZURE_OPENAI_ENDPOINT", "Enter your Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): ")
set_env_securely_azure("AZURE_OPENAI_DEPLOYMENT", "Enter your Azure OpenAI deployment name (e.g. gpt-4o): ")
print("RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.")

RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.


In [11]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")
    print("Row count:", cur.fetchone()[0])

    cur.execute("""
        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS
        FETCH FIRST 3 ROWS ONLY
    """)
    for row in cur.fetchall():
        print(row)

Row count: 1000
('0902.0428', 'Dynamics of planets in retrograde mean motion resonance', 'In a previous paper (Gayon &amp; Bois 2008a), we have shown the general efficiency of retrograde resonances for stabilizing compact planetary systems. Such retrograde resonances can be found when two-planets of a three-body planetary system are both in mean motion resonance and revolve in opposite directions. For a particular two-planet system, we have also obtained a new orbital fit involving such a counter-revolving configuration and consistent with the observational data. <br>In the present paper, we analytically investigate the three-body problem in this particular case of retrograde resonances. We therefore define a new set of canonical variables allowing to express correctly the resonance angles and obtain the Hamiltonian of a system harboring planets revolving in opposite directions. The acquiring of an analytical &#34;rail&#34; may notably contribute to a deeper understanding of our numeri

In [12]:
from agents import Agent, Runner

In [22]:
# Azure companion: configure openai-agents to use Azure OpenAI via DefaultAzureCredential
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from agents import Agent, set_default_openai_client, set_tracing_disabled

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

# Normalize endpoint in case env var includes /openai or deployment path.
raw_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint.lower():
    raw_endpoint = raw_endpoint[: raw_endpoint.lower().index("/openai")]

AZURE_OPENAI_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
# Runner uses Responses API; 2024-10-21 commonly fails on /responses in Azure.
env_api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
if not env_api_version or env_api_version == "2024-10-21":
    AZURE_OPENAI_API_VERSION = "2025-03-01-preview"
else:
    AZURE_OPENAI_API_VERSION = env_api_version

azure_agents_client = AsyncAzureOpenAI(
    azure_endpoint=raw_endpoint,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_ad_token_provider=token_provider,
    # Compatibility for openai<2.x credential gate when using only AAD token provider.
    _enforce_credentials=False,
)

# Guard: fail fast if this ever gets replaced with a sync client
if not isinstance(azure_agents_client, AsyncAzureOpenAI):
    raise TypeError(
        "Expected AsyncAzureOpenAI for agent runs. Restart kernel and rerun this cell before Azure runs."
    )

print(type(azure_agents_client))
print(f"Azure endpoint: {raw_endpoint}")
print(f"Azure deployment: {AZURE_OPENAI_MODEL}")
print(f"Azure API version: {AZURE_OPENAI_API_VERSION}")

# Route Agent/Runner calls to Azure OpenAI for the Azure companion cells
set_default_openai_client(azure_agents_client)
set_tracing_disabled(disabled=True)

research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)

<class 'openai.lib.azure.AsyncAzureOpenAI'>
Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4o
Azure API version: 2025-03-01-preview


In [14]:
from agents.tool import function_tool

@function_tool
def get_research_papers(user_query: str, retrieval_mode: str = "hybrid", top_k: int = 5) -> str:
    """
    Retrieves academic research papers relevant to the user's query.

    This tool queries the research_papers SQL table using one of three retrieval techniques:
        - 'keyword'  → lexical search via LIKE filtering
        - 'vector'   → semantic similarity search
        - 'hybrid'   → combines keyword prefiltering + vector similarity (default)

    Use this tool when analyzing or summarizing scientific literature.

    Args:
        user_query (str): Research topic or question to search for.
        retrieval_mode (str): 'keyword', 'vector', or 'hybrid'. Default is 'hybrid'.
        top_k (int): Number of top papers to retrieve (default=5).

    Returns:
        str: A formatted summary of the most relevant research papers.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using SQL-based functions (defined earlier)
    # ------------------------------------------------------------------
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
    else:
        rows, columns, _ = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=False
        )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format the output into a readable string
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No research papers found related to '{user_query}'."

    formatted_results = [f"📚 {retrieved_count} papers retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Paper")
        abstract = row_data.get("ABSTRACT", "No abstract available.")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] {title}\n"
            f"Abstract: {abstract}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


In [15]:
import pprint

### Build an Agent with Multiple Tool Access

In [16]:
from agents.tool import function_tool

@function_tool
def get_past_research_conversations(user_query: str, top_k: int = 5) -> str:
    """
    Retrieves relevant past research-related conversations or analyses related to the query.

    This tool searches a SQL database of prior research assistant conversations, 
    literature discussions, or synthesis sessions to find relevant context. 
    It allows the research assistant to recall previous analyses or summaries 
    that addressed similar topics, providing continuity and richer insights.

    Args:
        user_query (str): The research topic, concept, or question to search for.
        top_k (int): Number of top past discussions to retrieve (default=5).

    Returns:
        str: Formatted examples of relevant past research discussions.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using the SQL-based hybrid search (vector + keyword)
    # ------------------------------------------------------------------
    rows, columns, _ = hybrid_search_research_papers_pre_filter(
        conn=conn,
        embedding_model=embedding_model,
        search_phrase=user_query,
        top_k=top_k,
        show_explain=False
    )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format results for readability
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No past research discussions found related to '{user_query}'."

    formatted_results = [f"🧠 {retrieved_count} past research discussions retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Discussion")
        abstract = row_data.get("ABSTRACT", "No summary available.")
        snippet = row_data.get("TEXT_SNIPPET", "")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] **{title}**\n"
            f"Summary: {abstract}\n"
            f"Snippet: {snippet}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


Let's update our agent instruction to ensure it knows when to utilize the right tools

In [ ]:
# upgraded_research_paper_assistant = Agent(
#     name="Research Paper Assistant",
#     model=OPENAI_MODEL,
#     instructions="""
#     Always maintain an academic, evidence-based tone.
#     Your purpose is to help users explore, synthesize, and connect research insights —
#     not to speculate or fabricate information.
#     """,
# )


In [17]:
upgraded_research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
    Always maintain an academic, evidence-based tone.
    Your purpose is to help users explore, synthesize, and connect research insights —
    not to speculate or fabricate information.
    """,
)

In [ ]:
# # Attach research retrieval tools to the upgraded research assistant
# upgraded_research_paper_assistant.tools.append(get_research_papers)
# upgraded_research_paper_assistant.tools.append(get_past_research_conversations)

In [18]:
# Attach research retrieval tools to the upgraded Azure research assistant
upgraded_research_paper_assistant_azure.tools.append(get_research_papers)
upgraded_research_paper_assistant_azure.tools.append(get_past_research_conversations)

In [ ]:
# pprint.pprint(upgraded_research_paper_assistant.tools)

In [19]:
pprint.pprint(upgraded_research_paper_assistant_azure.tools)

[FunctionTool(name='get_research_papers',
              description='Retrieves academic research papers relevant to the '
                          "user's query.",
              params_json_schema={'additionalProperties': False,
                                  'properties': {'retrieval_mode': {'default': 'hybrid',
                                                                    'description': "'keyword', "
                                                                                   "'vector', "
                                                                                   'or '
                                                                                   "'hybrid'. "
                                                                                   'Default '
                                                                                   'is '
                                                                                   "'hybrid'.",
                        

In [ ]:
# run_result_with_tools = await Runner.run(
#     starting_agent=upgraded_research_paper_assistant,
#     input=(
#         "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
#     ),
# )

In [23]:
run_result_with_tools_azure = await Runner.run(
    starting_agent=upgraded_research_paper_assistant_azure,
    input=(
        "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
    ),
)

In [ ]:
# print(run_result_with_tools.raw_responses)

In [24]:
print(run_result_with_tools_azure.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"user_query":"rover navigation and planetary data collection","retrieval_mode":"hybrid","top_k":5}', call_id='call_DzRKwLsqQS0zFuE6LPafau62', name='get_research_papers', type='function_call', id='fc_039784f788b6e321006a1a070fc35c8193a891182b2312b5f0', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"mission planning and resource allocation for space exploration","retrieval_mode":"hybrid","top_k":5}', call_id='call_6oePSFmDADCDg6OljzFr6k6z', name='get_research_papers', type='function_call', id='fc_039784f788b6e321006a1a070fc36c81939b090f5ab7661a43', namespace=None, status='completed')], usage=Usage(requests=1, input_tokens=323, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=413, request_usage_entries=[]), response_id='resp_039784f788b6e321006a1a070d9c548193b08e39c4cfa1d53c', request_id=